<a href="https://colab.research.google.com/github/manavka/DeepFashion_DS207/blob/main/majority.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Set up

In [3]:
# Set up
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from google.colab import drive
drive.mount('/content/drive')

os.chdir("/content/drive/MyDrive/207_Final/DeepFashion/Category and Attribute Prediction Benchmark")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# Unzip images
import zipfile

img_dir = "/content/DeepFashion_img/img"
prepend_path = "/content/DeepFashion_img/" # use later
drive_img_dir = os.path.join(os.getcwd(), "Img") # where img.zip lives in Drive

if not os.path.exists(img_dir):
    with zipfile.ZipFile(os.path.join(drive_img_dir, "img.zip"), 'r') as zip_ref:
        zip_ref.extractall(prepend_path)

# Preprocessing: Anno_fine

In [14]:
# Data ingestion

category_lookup_df = pd.read_csv(
    "./Anno_fine/list_category_cloth.txt",
    sep=r"\s+",
    skiprows=1,
    header=0
)

category_lookup_df["key"] = category_lookup_df.index + 1

train_img_df = pd.read_csv(
    "./Anno_fine/train.txt",
    sep=r"\s+",
    names=["img_name"]
)

train_category_df = pd.read_csv(
    "./Anno_fine/train_cate.txt",
    sep=r"\s+",
    names=["category"]
)

val_img_df = pd.read_csv(
    "./Anno_fine/val.txt",
    sep=r"\s+",
    names=["img_name"]
)

val_category_df = pd.read_csv(
    "./Anno_fine/val_cate.txt",
    sep=r"\s+",
    names=["category"]
)

test_img_df = pd.read_csv(
    "./Anno_fine/test.txt",
    sep=r"\s+",
    names=["img_name"]
)

test_category_df = pd.read_csv(
    "./Anno_fine/test_cate.txt",
    sep=r"\s+",
    names=["category"]
)

In [19]:
# Create train and validation dataframes

train_df = pd.concat([train_img_df, train_category_df], axis=1)
train_df = train_df.merge(
    category_lookup_df,
    left_on="category",
    right_on="key",
    how="left"
)

val_df = pd.concat([val_img_df, val_category_df], axis=1)
val_df = val_df.merge(
    category_lookup_df,
    left_on="category",
    right_on="key",
    how="left"
)

test_df = pd.concat([test_img_df, test_category_df], axis=1)
test_df = test_df.merge(
    category_lookup_df,
    left_on="category",
    right_on="key",
    how="left"
)

train_df["category_type_tf"] = train_df["category_type"] - 1
val_df["category_type_tf"] = val_df["category_type"] - 1

train_df["img_path"] = train_df["img_name"].apply(lambda x: os.path.join(prepend_path, x))
val_df["img_path"] = val_df["img_name"].apply(lambda x: os.path.join(prepend_path, x))

test_df["category_type_tf"] = test_df["category_type"] - 1
test_df["img_path"] = test_df["img_name"].apply(lambda x: os.path.join(prepend_path, x))

label_names = ["upper body", "lower body", "full body"]

print(f"Train shape: {train_df.shape}")
print(f"Validation shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")
train_df.head()

Train shape: (14000, 7)
Validation shape: (2000, 7)
Test shape: (4000, 7)


,img_name,category,category_name,category_type,key,category_type_tf,img_path
0,img/Sweet_Crochet_Blouse/img_00000070.jpg,3,Blouse,1,3,0,/content/DeepFashion_img/img/Sweet_Crochet_Blo...
1,img/Classic_Pencil_Skirt/img_00000010.jpg,33,Skirt,2,33,1,/content/DeepFashion_img/img/Classic_Pencil_Sk...
2,img/Strapless_Diamond_Print_Dress/img_00000038...,41,Dress,3,41,2,/content/DeepFashion_img/img/Strapless_Diamond...
3,img/Mid-Rise_-_Acid_Wash_Skinny_Jeans/img_0000...,26,Jeans,2,26,1,/content/DeepFashion_img/img/Mid-Rise_-_Acid_W...
4,img/Zippered_Single-Button_Blazer/img_00000078...,2,Blazer,1,2,0,/content/DeepFashion_img/img/Zippered_Single-B...


In [20]:
# Add labels from the full dataset
Y_train = train_df["category_type_tf"].to_numpy()
Y_val = val_df["category_type_tf"].to_numpy()
Y_test = test_df["category_type_tf"].to_numpy()

print(f"Y_train shape: {Y_train.shape}")
print(f"Y_val shape: {Y_val.shape}")
print(f"Y_test shape: {Y_test.shape}")

Y_train shape: (14000,)
Y_val shape: (2000,)
Y_test shape: (4000,)


In [21]:
# Majority class classifier

class_counts = pd.Series(Y_train).value_counts().sort_index()
majority_class = class_counts.idxmax()

print("Class counts:")
for class_id, count in class_counts.items():
    print(f"{label_names[class_id]}: {count}")

print(f"\nMajority class: {label_names[majority_class]}\n")

Y_train_majority_pred = np.full_like(Y_train, majority_class)
Y_val_majority_pred = np.full_like(Y_val, majority_class)
Y_test_majority_pred = np.full_like(Y_test, majority_class)

train_majority_accuracy = accuracy_score(Y_train, Y_train_majority_pred)
val_majority_accuracy = accuracy_score(Y_val, Y_val_majority_pred)
test_majority_accuracy = accuracy_score(Y_test, Y_test_majority_pred)

print(f"Majority classifier train accuracy: {train_majority_accuracy:.4f}")
print(f"Majority classifier validation accuracy: {val_majority_accuracy:.4f}")
print(f"Majority classifier test accuracy: {test_majority_accuracy:.4f}")

Class counts:
upper body: 6372
lower body: 2644
full body: 4984

Majority class: upper body

Majority classifier train accuracy: 0.4551
Majority classifier validation accuracy: 0.4390
Majority classifier test accuracy: 0.4682
